In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia-uvm:     root         35 F.... nvidia-smi
/dev/nvidia0:        root         35 F.... nvidia-smi
/dev/nvidiactl:      root         35 F.... nvidia-smi


In [2]:
# Check GPU status
!nvidia-smi

Sun Mar 29 16:57:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

In [4]:
# For Kaggle setup
%pip install unsloth
%pip install --upgrade "torchvision>=0.25.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 97.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.1/188.1 MB 9.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.0 MB/s eta 0:00:

# Configuration

In [ ]:
SEED = 42

# Model configuration
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'

# Visual codebook configuration
K = 8192

BVIR_DATA_ID = 'alxxtexxr/BIVR-Data'
CODEBOOK_PATH = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'

In [6]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print(f"Random seed: {seed}")

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.10/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 Tesla P100-PCIE-16GB which is of compute capability (CC) 6.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/opt/conda/lib/python3.10/site-packages/torch/cuda/__init__.py:489: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA c

🦥 Unsloth Zoo will now patch everything to make training faster!
Random seed: 42


In [ ]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [8]:
import copy

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # False for LoRA 16bit
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # max_seq_length = 16384, # Must be this long for VLMs
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
    device_map = 'cpu', # Load in CPU first to not explode the memory usage in GPU when expanding the token embeddings
)
tokenizer_orig = copy.deepcopy(tokenizer) # Copy the original tokenizer for comparison later

print(model)

==((====))==  Unsloth 2026.3.17: Fast Qwen2_5_Vl patching. Transformers: 5.3.0.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 6.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_f

In [9]:
# Calculate LM embedding norms to set codebook scaling
E = model.get_input_embeddings().weight.data.clone()
lm_norms = E.norm(dim=1)
print("Embedding norms (mean, std):", lm_norms.mean().item(), lm_norms.std().item())

# Randomly sample K norms with replacement
target_norms = lm_norms[torch.randint(0, lm_norms.size(0), (K,))]
print("Target norms (mean, std):", target_norms.mean().item(), target_norms.std().item())

Embedding norms (mean, std): 0.8828125 0.2117919921875
Target norms (mean, std): 0.88037109375 0.2147216796875


In [10]:
def check_embs_info():
    print("Input embeddings shape:", model.get_input_embeddings().weight.shape)
    print("Output embeddings shape:", model.get_output_embeddings().weight.shape)
    print("Weight-tied:", model.get_input_embeddings().weight is model.get_output_embeddings().weight)

check_embs_info()

Input embeddings shape: torch.Size([152064, 3584])
Output embeddings shape: torch.Size([152064, 3584])
Weight-tied: False


In [11]:
list(tokenizer.tokenizer.get_added_vocab().items())[:30]

[('<|endoftext|>', 151643),
 ('<|im_start|>', 151644),
 ('<|im_end|>', 151645),
 ('<|object_ref_start|>', 151646),
 ('<|object_ref_end|>', 151647),
 ('<|box_start|>', 151648),
 ('<|box_end|>', 151649),
 ('<|quad_start|>', 151650),
 ('<|quad_end|>', 151651),
 ('<|vision_start|>', 151652),
 ('<|vision_end|>', 151653),
 ('<|vision_pad|>', 151654),
 ('<|image_pad|>', 151655),
 ('<|video_pad|>', 151656),
 ('<tool_call>', 151657),
 ('</tool_call>', 151658),
 ('<|fim_prefix|>', 151659),
 ('<|fim_middle|>', 151660),
 ('<|fim_suffix|>', 151661),
 ('<|fim_pad|>', 151662),
 ('<|repo_name|>', 151663),
 ('<|file_sep|>', 151664)]

In [12]:
def check_emb_stats(i):
    inner_model = getattr(model, 'model', model)
    x = inner_model.language_model.embed_tokens.weight[i].cpu().detach().numpy()
    print("i:", i)
    print(f"mean: {x.mean()}, max: {x.max()}, min: {x.min()}")
    print("First values:", x[:5])
    print()

check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding

i: 0
mean: 0.00015485286712646484, max: 0.12158203125, min: -0.1826171875
First values: [-0.01709  -0.001259  0.01892   0.01361   0.02698 ]

i: 151664
mean: 0.0, max: 0.0, min: 0.0
First values: [ 0. -0.  0. -0. -0.]



In [ ]:
vtoks = [f'<|vtok_{i}|>' for i in range(K)]
tokenizer.tokenizer.add_tokens(vtoks);

In [14]:
vtok_start_id, vtok_end_id = tokenizer.tokenizer.encode(f'<|vtok_0|><|vtok_{K-1}|>')
print("Visual token start ID:", vtok_start_id)
print("Visual token end ID:", vtok_end_id)

Visual token start ID: 151665
Visual token end ID: 159856


In [15]:
model.resize_token_embeddings(vtok_end_id+1, mean_resizing=False)
print(model)

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_f

In [16]:
model = model.to("cuda")
torch.cuda.empty_cache()
# TODO: Clear the CPU memory too

In [17]:
check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding
check_emb_stats(vtok_start_id) # Check the stats of the first virtual token embedding
check_emb_stats(vtok_end_id) # Check the stats of the last virtual token embedding

i: 0
mean: 0.00015485286712646484, max: 0.12158203125, min: -0.1826171875
First values: [-0.01709  -0.001259  0.01892   0.01361   0.02698 ]

i: 151664
mean: 0.0, max: 0.0, min: 0.0
First values: [ 0. -0.  0. -0. -0.]

i: 151665
mean: 0.0, max: -0.0, min: -0.0
First values: [-0. -0. -0.  0.  0.]

i: 159856
mean: 0.0003743171691894531, max: 0.0743408203125, min: -0.0704345703125
First values: [ 0.011665 -0.00489  -0.01104  -0.0438   -0.02467 ]



In [18]:
from huggingface_hub import hf_hub_download

codebook_path = hf_hub_download(
    repo_id=BVIR_DATA_ID,
    filename=CODEBOOK_PATH,
    repo_type='dataset',
)
print("Visual codebook downloaded to:", codebook_path)

model_unsloth_Qwen2.5-VL-7B-Instruct-bnb(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Visual codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BIVR-Data/snapshots/5de7da8ec1b2cd2138e50bc11b18e17d76cbf942/model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy


In [19]:
import numpy as np

codebook = torch.from_numpy(np.load(codebook_path)).to(torch.float16) # (K, 3584) -> Current: (8192, 3584)
assert codebook.shape[0] == K, f"Expected codebook with {K} entries, but got {codebook.shape[0]}"

print(f"Visual codebook shape: {codebook.shape}, dtype: {codebook.dtype}")
print("Visual codebook norms before scaling (mean, std):", codebook.norm(dim=1).mean().item(), codebook.norm(dim=1).std().item())

Visual codebook shape: torch.Size([8192, 3584]), dtype: torch.float16
Visual codebook norms before scaling (mean, std): 0.798828125 0.069580078125


In [20]:
# Normalize codebook to unit vectors first
codebook_norm = codebook / codebook.norm(dim=1, keepdim=True) 
print("Visual codebook norms after normalization (mean, std):", codebook_norm.norm(dim=1).mean().item(), codebook_norm.norm(dim=1).std().item())

# Scale to LM-like norms
codebook_scaled = codebook_norm * target_norms[:, None]              
print("Visual codebook norms after scaling (mean, std):", codebook_scaled.norm(dim=1).mean().item(), codebook_scaled.norm(dim=1).std().item())

Visual codebook norms after normalization (mean, std): 1.0 0.00014650821685791016
Visual codebook norms after scaling (mean, std): 0.88037109375 0.2147216796875


In [21]:
chunk_size = 512

with torch.no_grad():
    for i in range(0, vtok_end_id - vtok_start_id + 1, chunk_size):
        a_start = vtok_start_id + i
        a_end = min(a_start + chunk_size, vtok_end_id + 1)

        b_start = i
        b_end = b_start + (a_end - a_start)
        
        print(f"Processing chunk: codebook_scaled[{b_start}:{b_end}] => embed_tokens[{a_start}:{a_end}]")

        # Move only a chunk to GPU
        chunk = codebook_scaled[b_start:b_end].to(
            model.lm_head.weight.device,
            dtype=model.lm_head.weight.dtype,
            non_blocking=True
        )

        # Replace in the input embeddings
        getattr(model, 'model', model).language_model.embed_tokens.weight[a_start:a_end] = chunk
        
        # Replace in the output embeddings too even if they are not tied, to give better initialization
        model.lm_head.weight[a_start:a_end] = chunk

        # Optional: free reference quickly
        del chunk

Processing chunk: codebook_scaled[0:512] => embed_tokens[151665:152177]
Processing chunk: codebook_scaled[512:1024] => embed_tokens[152177:152689]
Processing chunk: codebook_scaled[1024:1536] => embed_tokens[152689:153201]
Processing chunk: codebook_scaled[1536:2048] => embed_tokens[153201:153713]
Processing chunk: codebook_scaled[2048:2560] => embed_tokens[153713:154225]
Processing chunk: codebook_scaled[2560:3072] => embed_tokens[154225:154737]
Processing chunk: codebook_scaled[3072:3584] => embed_tokens[154737:155249]
Processing chunk: codebook_scaled[3584:4096] => embed_tokens[155249:155761]
Processing chunk: codebook_scaled[4096:4608] => embed_tokens[155761:156273]
Processing chunk: codebook_scaled[4608:5120] => embed_tokens[156273:156785]
Processing chunk: codebook_scaled[5120:5632] => embed_tokens[156785:157297]
Processing chunk: codebook_scaled[5632:6144] => embed_tokens[157297:157809]
Processing chunk: codebook_scaled[6144:6656] => embed_tokens[157809:158321]
Processing chunk:

In [22]:
model.get_input_embeddings().weight is model.get_output_embeddings().weight

False

In [23]:
check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding
check_emb_stats(152063) # Check the stats of the last embedding in the vocabulary
check_emb_stats(vtok_start_id) # Check the stats of the first virtual token embedding
check_emb_stats(vtok_end_id) # Check the stats of the last virtual token embedding

i: 0
mean: 0.00015485286712646484, max: 0.12158203125, min: -0.1826171875
First values: [-0.01709  -0.001259  0.01892   0.01361   0.02698 ]

i: 151664
mean: 0.0, max: 0.0, min: 0.0
First values: [ 0. -0.  0. -0. -0.]

i: 152063
mean: -0.00014090538024902344, max: 0.331298828125, min: -0.1104736328125
First values: [ 0.009224  0.002768  0.00521   0.0324   -0.006287]

i: 151665
mean: -0.00011551380157470703, max: 0.265625, min: -0.1484375
First values: [ 0.01587   0.010506 -0.002207  0.002174 -0.00622 ]

i: 159856
mean: 2.6524066925048828e-05, max: 0.28564453125, min: -0.1063232421875
First values: [ 0.014885   0.01523   -0.0007777 -0.00788    0.00723  ]



In [24]:
# Verify that the added embeddings match the scaled codebook
embed_tokens_added = getattr(model, 'model', model).language_model.embed_tokens.weight[-K:].cpu().detach()
lm_head_added = model.lm_head.weight[-K:].cpu().detach()

assert torch.allclose(codebook_scaled, embed_tokens_added) and torch.allclose(codebook_scaled, lm_head_added), "The added embeddings do not match the scaled codebook!"

def check_stats(x):
    print(f"mean: {x.mean().item()}, max: {x.max().item()}, min: {x.min().item()}")

print("Last scaled codebook embedding stats:")
check_stats(codebook_scaled.numpy()[-1])
print()
print("Last input embedding stats:")
check_stats(embed_tokens_added.numpy()[-1])
print()
print("Last output embedding stats:")
check_stats(lm_head_added.numpy()[-1])

Last scaled codebook embedding stats:
mean: 2.6524066925048828e-05, max: 0.28564453125, min: -0.1063232421875

Last input embedding stats:
mean: 2.6524066925048828e-05, max: 0.28564453125, min: -0.1063232421875

Last output embedding stats:
mean: 2.6524066925048828e-05, max: 0.28564453125, min: -0.1063232421875


In [25]:
model.config.vocab_size = len(tokenizer.tokenizer)
print("Updated model config vocab size:", model.config.vocab_size)

Updated model config vocab size: 159857


In [26]:
HUB_MODEL_ID = 'alxxtexxr/Qwen2.5-VL-7B-Instruct-bnb-4bit-with-codebook'

model.push_to_hub(
    HUB_MODEL_ID,
    max_shard_size='6GB'
)

tokenizer.push_to_hub(HUB_MODEL_ID)

README.md:   0%|          | 0.00/561 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/alxxtexxr/Qwen2.5-VL-7B-Instruct-bnb-4bit-with-codebook


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            